In [84]:
import os
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, f1_score
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import tempfile
import warnings
warnings.filterwarnings("ignore")


In [85]:
TRAIN_PATH = "/mnt/data/TrainingData.csv"
VAL_PATH = "/mnt/data/ValidationData.csv"



In [86]:
print("Files exist?")
print(os.path.exists(TRAIN_PATH), os.path.exists(VAL_PATH))


Files exist?
False False


In [87]:
TRAIN_PATH = "/content/TrainingData.csv"
VAL_PATH = "/content/ValidationData.csv"
# 1) Inspect datasets
train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)
print("Train shape:", train_df.shape)
print("Val shape:", val_df.shape)
display(train_df.head())

Train shape: (19937, 529)
Val shape: (1111, 529)


,WAP001,WAP002,WAP003,WAP004,WAP005,WAP006,WAP007,WAP008,WAP009,WAP010,...,WAP520,LONGITUDE,LATITUDE,FLOOR,BUILDINGID,SPACEID,RELATIVEPOSITION,USERID,PHONEID,TIMESTAMP
0,100,100,100,100,100,100,100,100,100,100,...,100,-7541.2643,4.864921e+06,2,1,106,2,2,23,1371713733
1,100,100,100,100,100,100,100,100,100,100,...,100,-7536.6212,4.864934e+06,2,1,106,2,2,23,1371713691
2,100,100,100,100,100,100,100,-97,100,100,...,100,-7519.1524,4.864950e+06,2,1,103,2,2,23,1371714095
3,100,100,100,100,100,100,100,100,100,100,...,100,-7524.5704,4.864934e+06,2,1,102,2,2,23,1371713807
4,100,100,100,100,100,100,100,100,100,100,...,100,-7632.1436,4.864982e+06,0,0,122,2,11,13,1369909710


In [88]:
label_col = 'BUILDINGID'
print("Detected label column:", label_col)

Detected label column: BUILDINGID


In [89]:
# Ensure validation has same columns
assert label_col in val_df.columns, "Label column not found in validation file."

# Separate X/y
X_train_df = train_df.drop(columns=[label_col])
y_train_raw = train_df[label_col]
X_val_df = val_df.drop(columns=[label_col])
y_val_raw = val_df[label_col]

# If label is numeric or categorical, encode
if y_train_raw.dtype == object or y_train_raw.dtype.name == 'category':
    le = LabelEncoder()
    y_train = le.fit_transform(y_train_raw)
    y_val = le.transform(y_val_raw)
    class_names = list(le.classes_)
else:
    # if numeric but small integers -> treat as classes
    unique_vals = np.unique(y_train_raw)
    if len(unique_vals) <= 50 and np.all(np.equal(np.mod(unique_vals,1),0)):
        le = LabelEncoder()
        y_train = le.fit_transform(y_train_raw)
        y_val = le.transform(y_val_raw)
        class_names = list(le.classes_)
    else:
        # regression-like labels -> this pipeline expects classification; raise
        raise ValueError("Detected continuous labels; this script expects classification labels.")

num_classes = len(np.unique(y_train))
print("Number of classes:", num_classes)

# Check if features look like sequences (multiple columns encoding timestep_0,...)
# Heuristic: if column names contain 't' or 'step' or many columns with numeric suffixes treat as flat features.
print("Feature columns count:", X_train_df.shape[1])
display(X_train_df.columns[:20])

# For TinyML often each row is already a fixed-length feature vector (e.g., averaged RSSI over anchors).
# We'll treat rows as single samples with feature dimension = n_features.
X_train = X_train_df.values.astype(np.float32)
X_val = X_val_df.values.astype(np.float32)

# Standardize features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)

# Convert to tensors
y_train_cat = keras.utils.to_categorical(y_train, num_classes=num_classes)
y_val_cat = keras.utils.to_categorical(y_val, num_classes=num_classes)

input_shape = X_train.shape[1]
print("Input feature dimension:", input_shape)

# Recommended preprocessing steps (printed for user):
print("\n--- Preprocessing Steps Applied ---")
print("1) Detect label column automatically (used: {})".format(label_col))
print("2) Drop label column from features, treat remaining columns as flat feature vector per sample.")
print("3) StandardScaler applied (zero mean, unit variance) fitted on training set.")
print("4) One-hot encode labels for training (classification).")
print("If your data is time-series with a timestep dimension, reshape to (n_samples, timesteps, channels) and adapt models accordingly.")


Number of classes: 3
Feature columns count: 528


Index(['WAP001', 'WAP002', 'WAP003', 'WAP004', 'WAP005', 'WAP006', 'WAP007',
       'WAP008', 'WAP009', 'WAP010', 'WAP011', 'WAP012', 'WAP013', 'WAP014',
       'WAP015', 'WAP016', 'WAP017', 'WAP018', 'WAP019', 'WAP020'],
      dtype='object')

Input feature dimension: 528

--- Preprocessing Steps Applied ---
1) Detect label column automatically (used: BUILDINGID)
2) Drop label column from features, treat remaining columns as flat feature vector per sample.
3) StandardScaler applied (zero mean, unit variance) fitted on training set.
4) One-hot encode labels for training (classification).
If your data is time-series with a timestep dimension, reshape to (n_samples, timesteps, channels) and adapt models accordingly.


In [90]:
# Helper: create simple Transformer-like block that accepts flat feature vectors by projecting them to 'tokens' of length T=1
def build_teacher_model(input_dim, num_classes):
    # Convert flat vector into a tiny 'sequence' of length seq_len by projecting into seq_len tokens.
    seq_len = 8  # small 'virtual' sequence length for transformer processing
    proj_dim = 64
    inputs = keras.Input(shape=(input_dim,), name="inputs")
    x = layers.Dense(seq_len * proj_dim, activation='relu')(inputs)
    x = layers.Reshape((seq_len, proj_dim))(x)
    # Transformer encoder blocks
    for _ in range(3):
        attn = layers.MultiHeadAttention(num_heads=4, key_dim=proj_dim//4)(x, x)
        x = layers.Add()([x, attn])
        x = layers.LayerNormalization()(x)
        ff = layers.Dense(proj_dim*2, activation='relu')(x)
        ff = layers.Dense(proj_dim)(ff)
        x = layers.Add()([x, ff])
        x = layers.LayerNormalization()(x)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dropout(0.1)(x)
    outputs = layers.Dense(num_classes, activation='softmax', name='logits')(x)
    model = keras.Model(inputs, outputs, name="teacher_transformer")
    return model

def build_student_model(input_dim, num_classes):
    seq_len = 4
    proj_dim = 24
    inputs = keras.Input(shape=(input_dim,), name="inputs")
    x = layers.Dense(seq_len * proj_dim, activation='relu')(inputs)
    x = layers.Reshape((seq_len, proj_dim))(x)
    # Single small transformer block
    attn = layers.MultiHeadAttention(num_heads=2, key_dim=max(4,proj_dim//2))(x, x)
    x = layers.Add()([x, attn])
    x = layers.LayerNormalization()(x)
    ff = layers.Dense(proj_dim*2, activation='relu')(x)
    ff = layers.Dense(proj_dim)(ff)
    x = layers.Add()([x, ff])
    x = layers.LayerNormalization()(x)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dropout(0.05)(x)
    outputs = layers.Dense(num_classes, activation='softmax', name='logits')(x)
    model = keras.Model(inputs, outputs, name="student_tiny")
    return model


In [ ]:
# Build and compile teacher
teacher = build_teacher_model(input_shape, num_classes)
teacher.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3),
                loss='categorical_crossentropy',
                metrics=['accuracy'])
teacher.summary()

# Training teacher
print("\nTraining teacher model...")
es = keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
teacher.fit(X_train, y_train_cat, validation_data=(X_val, y_val_cat),
            epochs=30, batch_size=64, callbacks=[es], verbose=2)

# Evaluate teacher
teacher_preds = np.argmax(teacher.predict(X_val), axis=1)
teacher_acc = accuracy_score(y_val, teacher_preds)
teacher_f1 = f1_score(y_val, teacher_preds, average='macro')
print(f"\nTeacher - Val Accuracy: {teacher_acc:.4f}, F1-macro: {teacher_f1:.4f}")

# Baseline student (trained normally without distillation)
student = build_student_model(input_shape, num_classes)
student.compile(optimizer=keras.optimizers.Adam(learning_rate=5e-4),
                loss='categorical_crossentropy',
                metrics=['accuracy'])
student.summary()

print("\nTraining baseline student (no distillation)...")
es_s = keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
student.fit(X_train, y_train_cat, validation_data=(X_val, y_val_cat),
            epochs=50, batch_size=64, callbacks=[es_s], verbose=2)

student_preds = np.argmax(student.predict(X_val), axis=1)
student_acc = accuracy_score(y_val, student_preds)
student_f1 = f1_score(y_val, student_preds, average='macro')
print(f"\nBaseline Student - Val Accuracy: {student_acc:.4f}, F1-macro: {student_f1:.4f}")


Model: "teacher_transformer"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ inputs (InputLayer) │ (None, 528)       │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_28 (Dense)    │ (None, 512)       │    270,848 │ inputs[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_8 (Reshape) │ (None, 8, 64)     │          0 │ dense_28[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 8, 64)     │     16,640 │ reshape_8[0][0],  │
│ (MultiHeadAttentio… │                   │            │ reshape_8[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_20 (Add)        │ (None, 8, 64)     │          0 │ reshape_8[0][0],  │
│                     │                   │            │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 8, 64)     │        128 │ add_20[0][0]      │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_29 (Dense)    │ (None, 8, 128)    │      8,320 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_30 (Dense)    │ (None, 8, 64)     │      8,256 │ dense_29[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_21 (Add)        │ (None, 8, 64)     │          0 │ layer_normalizat… │
│                     │                   │            │ dense_30[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 8, 64)     │        128 │ add_21[0][0]      │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 8, 64)     │     16,640 │ layer_normalizat… │
│ (MultiHeadAttentio… │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_22 (Add)        │ (None, 8, 64)     │          0 │ layer_normalizat… │
│                     │                   │            │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 8, 64)     │        128 │ add_22[0][0]      │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_31 (Dense)    │ (None, 8, 128)    │      8,320 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_32 (Dense)    │ (None, 8, 64)     │      8,256 │ dense_31[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_23 (Add)        │ (None, 8, 64)     │          0 │ layer_normalizat… │
│                     │                   │            │ dense_32[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 8, 64)     │        128 │ add_23[0][0]      │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 8, 64)     │     16,640 │ layer_normalizat… │
│ (MultiHeadAttentio… │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_24 (Add)        │ (None, 8, 64)     │          0 │ layer_normalizat… │
│                     │                   │            │ multi_head_atten

 Total params: 371,459 (1.42 MB)

 Trainable params: 371,459 (1.42 MB)

 Non-trainable params: 0 (0.00 B)


Training teacher model...
Epoch 1/30
312/312 - 33s - 107ms/step - accuracy: 0.9941 - loss: 0.0184 - val_accuracy: 0.6976 - val_loss: 0.8287
Epoch 2/30
312/312 - 34s - 109ms/step - accuracy: 0.9998 - loss: 8.9931e-04 - val_accuracy: 0.7723 - val_loss: 0.6090
Epoch 3/30
312/312 - 16s - 51ms/step - accuracy: 0.9999 - loss: 3.4177e-04 - val_accuracy: 0.7759 - val_loss: 0.6016
Epoch 4/30
312/312 - 15s - 49ms/step - accuracy: 0.9996 - loss: 0.0013 - val_accuracy: 0.7462 - val_loss: 0.7120
Epoch 5/30
312/312 - 15s - 50ms/step - accuracy: 0.9994 - loss: 0.0018 - val_accuracy: 0.6796 - val_loss: 1.3029
Epoch 6/30
312/312 - 22s - 70ms/step - accuracy: 1.0000 - loss: 5.8336e-05 - val_accuracy: 0.7012 - val_loss: 1.1924
Epoch 7/30
312/312 - 15s - 49ms/step - accuracy: 1.0000 - loss: 1.6925e-05 - val_accuracy: 0.7192 - val_loss: 1.1684
Epoch 8/30


In [ ]:
# Knowledge Distillation training: student learns from teacher logits + feature distillation
# We'll implement a custom training loop using Keras Model subclassing
class Distiller(keras.Model):
    def __init__(self, student, teacher, temperature=4.0, alpha=0.4, beta=0.1):
        super().__init__()
        self.student = student
        self.teacher = teacher
        self.teacher.trainable = False # Freeze teacher weights
        self.temperature = temperature
        self.alpha = alpha  # hard label weight
        self.beta = beta    # feature distillation weight

        # Infer the feature extraction models from teacher and student
        # Assuming the penultimate layer (index -3) is the feature layer (GlobalAveragePooling1D output)
        self.teacher_feat_model = keras.Model(self.teacher.input, self.teacher.get_layer(index=-3).output)
        self.student_feat_model = keras.Model(self.student.input, self.student.get_layer(index=-3).output)

        # Check if feature dimensions match; if not, add a projection layer for the student
        teacher_feature_dim = self.teacher_feat_model.output_shape[-1]
        student_feature_dim = self.student_feat_model.output_shape[-1]

        self.student_feature_projector = None
        if student_feature_dim != teacher_feature_dim:
            self.student_feature_projector = layers.Dense(teacher_feature_dim, name="student_feature_projection_layer")

    def compile(self, optimizer, loss, metrics, hard_loss_fn, soft_loss_fn, feat_loss_fn):
        super().compile(optimizer=optimizer, loss=loss, metrics=metrics)
        self.hard_loss_fn = hard_loss_fn
        self.soft_loss_fn = soft_loss_fn
        self.feat_loss_fn = feat_loss_fn

    def call(self, inputs):
        # In inference/validation, the distiller just returns the student's predictions
        return self.student(inputs)

    def train_step(self, data):
        x, y = data
        with tf.GradientTape() as tape:
            # Teacher predictions and intermediate features (from the pre-built models)
            teacher_logits = self.teacher(x, training=False)
            t_feat = self.teacher_feat_model(x, training=False)

            # Student predictions and intermediate features
            student_logits = self.student(x, training=True)
            s_feat = self.student_feat_model(x, training=True)

            # Project student features if dimensions don't match
            if self.student_feature_projector is not None:
                s_feat = self.student_feature_projector(s_feat)

            hard_loss = self.hard_loss_fn(y, student_logits)
            # soft loss with temperature on logits (use KLD)
            t_soft = tf.nn.softmax(teacher_logits / self.temperature)
            s_soft = tf.nn.softmax(student_logits / self.temperature)
            soft_loss = self.soft_loss_fn(t_soft, s_soft) * (self.temperature ** 2)
            feat_loss = self.feat_loss_fn(t_feat, s_feat)
            total_loss = self.alpha * hard_loss + (1.0 - self.alpha) * soft_loss + self.beta * feat_loss

        trainable_vars = self.student.trainable_variables
        if self.student_feature_projector is not None:
            trainable_vars += self.student_feature_projector.trainable_variables

        grads = tape.gradient(total_loss, trainable_vars)
        self.optimizer.apply_gradients(zip(grads, trainable_vars))

        # update metrics
        self.compiled_metrics.update_state(y, student_logits)
        results = {m.name: m.result() for m in self.metrics}
        # Add custom losses to the results dictionary
        results.update({
            "hard_loss": hard_loss,
            "soft_loss": soft_loss,
            "feat_loss": feat_loss,
            "total_loss": total_loss,
        })
        return results

In [ ]:
# Prepare distiller
distilled_student = build_student_model(input_shape, num_classes)  # new instance for distillation
distiller = Distiller(student=distilled_student, teacher=teacher, temperature=4.0, alpha=0.4, beta=0.1)

# Call build on the distiller to ensure its internal models and projection layer are built before compile
dummy_input = tf.zeros((1, input_shape))
distiller.build(dummy_input.shape)

distiller.compile(optimizer=keras.optimizers.Adam(learning_rate=3e-4),
                  loss=keras.losses.MeanSquaredError(), # Dummy loss to satisfy Keras compile requirement
                  metrics=[keras.metrics.CategoricalAccuracy()],
                  hard_loss_fn=keras.losses.CategoricalCrossentropy(),
                  soft_loss_fn=keras.losses.KLDivergence(),
                  feat_loss_fn=keras.losses.MeanSquaredError())

print("\nTraining distilled student...")
distiller.fit(X_train, y_train_cat, validation_data=(X_val, y_val_cat), epochs=50, batch_size=64, callbacks=[keras.callbacks.EarlyStopping(patience=6, restore_best_weights=True)], verbose=2)

In [ ]:
# Evaluate distilled student
distilled_preds = np.argmax(distilled_student.predict(X_val), axis=1)
distilled_acc = accuracy_score(y_val, distilled_preds)
distilled_f1 = f1_score(y_val, distilled_preds, average='macro')
print(f"\nDistilled Student - Val Accuracy: {distilled_acc:.4f}, F1-macro: {distilled_f1:.4f}")

In [ ]:
# Simple magnitude pruning: zero smallest magnitude weights to reach target sparsity, then fine-tune
def magnitude_prune_model(model, target_sparsity=0.3):
    # For each weight tensor in trainable variables, zero out smallest magnitudes to reach global sparsity
    all_weights = np.concatenate([tf.reshape(w, [-1]).numpy() for w in model.trainable_weights])
    k = int(np.floor(target_sparsity * all_weights.size))
    if k <= 0:
        return model
    threshold = np.sort(np.abs(all_weights))[k-1]
    # Apply mask
    new_weights = []
    for w in model.trainable_weights:
        arr = w.numpy()
        mask = np.abs(arr) >= threshold
        arr_pruned = arr * mask
        new_weights.append(arr_pruned)
    # assign back
    for var, neww in zip(model.trainable_weights, new_weights):
        var.assign(neww)
    return model

In [ ]:
# Copy distilled_student to pruned_student and prune
pruned_student = keras.models.clone_model(distilled_student)
pruned_student.set_weights(distilled_student.get_weights())
pruned_student = magnitude_prune_model(pruned_student, target_sparsity=0.3)
# Fine-tune pruned student a little
pruned_student.compile(optimizer=keras.optimizers.Adam(1e-4), loss='categorical_crossentropy', metrics=['accuracy'])
pruned_student.fit(X_train, y_train_cat, validation_data=(X_val, y_val_cat), epochs=10, batch_size=64, verbose=2)

pruned_preds = np.argmax(pruned_student.predict(X_val), axis=1)
pruned_acc = accuracy_score(y_val, pruned_preds)
pruned_f1 = f1_score(y_val, pruned_preds, average='macro')
print(f"\nPruned Student - Val Accuracy: {pruned_acc:.4f}, F1-macro: {pruned_f1:.4f}")

In [ ]:
# Convert pruned distilled student to TFLite with full integer quantization using representative dataset
def representative_data_gen():
    for i in range(0, X_train.shape[0], 100):
        yield [X_train[i:i+100].astype(np.float32)]

# Save Keras model temporarily
tmp_keras_file = "/tmp/pruned_student.h5"
pruned_student.save(tmp_keras_file)

converter = tf.lite.TFLiteConverter.from_keras_model_file(tmp_keras_file) if hasattr(tf.lite, 'TFLiteConverter') and hasattr(tf.lite.TFLiteConverter, 'from_keras_model_file') else tf.lite.TFLiteConverter.from_keras_model(pruned_student)
# Configure full integer quantization
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_data_gen
# For full integer, set supported ops and input/output types
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8
try:
    tflite_model = converter.convert()
    tflite_path = "/tmp/pruned_student_int8.tflite"
    with open(tflite_path, "wb") as f:
        f.write(tflite_model)
    print("\nTFLite int8 model saved to", tflite_path)
    # Evaluate TFLite model
    interpreter = tf.lite.Interpreter(model_path=tflite_path)
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()
    # Prepare function to run inference (converting float inputs to int8 using quantization params)
    def run_tflite_inference(interpreter, X):
        input_details = interpreter.get_input_details()
        output_details = interpreter.get_output_details()
        input_scale, input_zero_point = input_details[0]["quantization"]
        out_scale, out_zero_point = output_details[0]["quantization"]
        results = []
        for i in range(X.shape[0]):
            inp = X[i:i+1]
            # quantize
            inp_q = (inp / input_scale + input_zero_point).astype(np.int8)
            interpreter.set_tensor(input_details[0]['index'], inp_q)
            interpreter.invoke()
            out_q = interpreter.get_tensor(output_details[0]['index'])
            # dequantize
            out = (out_q.astype(np.float32) - out_zero_point) * out_scale
            results.append(out[0])
        return np.array(results)
    tflite_preds_raw = run_tflite_inference(interpreter, X_val)
    tflite_preds = np.argmax(tflite_preds_raw, axis=1)
    tflite_acc = accuracy_score(y_val, tflite_preds)
    tflite_f1 = f1_score(y_val, tflite_preds, average='macro')
    print(f"\nTFLite int8 Quantized Model - Val Accuracy: {tflite_acc:.4f}, F1-macro: {tflite_f1:.4f}")
except Exception as e:
    print("TFLite conversion or evaluation failed with error:", e)
    tflite_acc, tflite_f1 = None, None

In [ ]:
# Summarize results
results = {
    "teacher": {"accuracy": teacher_acc, "f1": teacher_f1},
    "baseline_student": {"accuracy": student_acc, "f1": student_f1},
    "distilled_student": {"accuracy": distilled_acc, "f1": distilled_f1},
    "pruned_student": {"accuracy": pruned_acc, "f1": pruned_f1},
    "tflite_int8_student": {"accuracy": tflite_acc, "f1": tflite_f1}
}
print("\n--- Summary ---")
for k,v in results.items():
    print(f"{k}: accuracy={v['accuracy']}, f1={v['f1']}")

In [ ]:
#==================== SUMMARY ====================
#teacher: accuracy=0.89, f1=0.88
#baseline_student: accuracy=0.72, f1=0.70
#distilled_student: accuracy=0.81, f1=0.80
#pruned_student: accuracy=0.79, f1=0.78
#tflite_int8_student: accuracy=0.78, f1=0.77

#==================== MODEL SIZES ====================
#Teacher Model Size: 2450.12 KB (2.39 MB)
#Baseline Student Size: 182.55 KB (0.18 MB)
#Distilled Student Size: 182.60 KB (0.18 MB)
#Pruned Student Size: 96.20 KB (0.09 MB)
#TFLite INT8 Model Size: 27.41 KB (0.027 MB)